In [ ]:
import pandas as pd
import os, sys
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, GradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

ruta_raiz = os.path.abspath('..')
if ruta_raiz not in sys.path:
    sys.path.append(ruta_raiz)

%load_ext autoreload
%autoreload 2
from Data_preprocesing.IQRCapper import IQRCapper


X_train = pd.read_csv("../Train_test_split/X_train.csv")
y_train = pd.read_csv("../Train_test_split/y_train.csv").values.ravel()
X_test = pd.read_csv("../Train_test_split/X_test.csv")
y_test = pd.read_csv("../Train_test_split/y_test.csv").values.ravel()

target_folder = '../comparations'
os.makedirs(target_folder, exist_ok=True) 


def save_experiment(model_name, imbalance_method, accuracy, precision, recall, f1, roc_auc):
    new_result = {
        'Model': [model_name],
        'Imbalance_Method': [imbalance_method],
        'Accuracy': [round(accuracy, 4)],
        'Precision': [round(precision, 4)],
        'Recall': [round(recall, 4)],
        'F1_Score': [round(f1, 4)],
        'ROC_AUC': [round(roc_auc, 4)]
    }
    df_new = pd.DataFrame(new_result)
    
    
    csv_file = f'{target_folder}/imbalance_results.csv'
    
    if os.path.exists(csv_file):
        df_new.to_csv(csv_file, mode='a', header=False, index=False)
    else:
        df_new.to_csv(csv_file, index=False)
        
    print(f"✅ Success! Results for {model_name} + {imbalance_method} saved in the '{target_folder}' folder.")

C:\Users\alvarodela.herran\AppData\Roaming\Python\Python312\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\alvarodela.herran\AppData\Roaming\Python\Python312\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


Voting classifier

In [4]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler, SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler, TomekLinks
from imblearn.combine import SMOTETomek, SMOTEENN
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def build_voting_classifier():
    lr  = LogisticRegression(random_state=42, max_iter=1000)
    rfc = RandomForestClassifier(random_state=42)
    svc = SVC(probability=True, random_state=42)
    return VotingClassifier(
        estimators=[('lr', lr), ('rfc', rfc), ('svc', svc)],
        voting='soft'
    )

imbalance_methods = {
    "Baseline": None,
    "RandomUnderSampler":     RandomUnderSampler(random_state=42),
    "TomekLinks":             TomekLinks(),
    "RandomOverSampler":      RandomOverSampler(random_state=42),
    "SMOTE":                  SMOTE(random_state=42),
    "ADASYN":                 ADASYN(random_state=42),
    "SMOTETomek":             SMOTETomek(random_state=42),
    "SMOTEENN":               SMOTEENN(random_state=42),
}

for method_name, sampler in imbalance_methods.items():
    print(f"\n--- {method_name} ---")

    voting_clf = build_voting_classifier()

    if sampler is None:
        pipe = ImbPipeline([
            ('capping_outliers', IQRCapper(factor=1.5)),
            ('imputacion', SimpleImputer(strategy='mean')),
            ('scaler', StandardScaler()),
            ('classifier', voting_clf)])
    else:
        pipe = ImbPipeline([
            ('capping_outliers', IQRCapper(factor=1.5)),
            ('imputacion', SimpleImputer(strategy='mean')),
            ('scaler', StandardScaler()),
            ('sampler', sampler), ('classifier', voting_clf)])

    pipe.fit(X_train, y_train)

    y_pred  = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)

    accuracy  = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall    = recall_score(y_test, y_pred, average='weighted')
    f1        = f1_score(y_test, y_pred, average='weighted')
    roc_auc   = roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted')

    print(f"Accuracy: {accuracy:.4f} | F1: {f1:.4f} | ROC-AUC: {roc_auc:.4f}")

    save_experiment(
        model_name="VotingClassifier (LR+RFC+SVC)",
        imbalance_method=method_name,
        accuracy=accuracy,
        precision=precision,
        recall=recall,
        f1=f1,
        roc_auc=roc_auc
    )


--- Baseline ---
Accuracy: 0.7492 | F1: 0.7450 | ROC-AUC: 0.8964
✅ Success! Results for VotingClassifier (LR+RFC+SVC) + Baseline saved in the '../comparations' folder.

--- RandomUnderSampler ---
Accuracy: 0.7182 | F1: 0.7209 | ROC-AUC: 0.8918
✅ Success! Results for VotingClassifier (LR+RFC+SVC) + RandomUnderSampler saved in the '../comparations' folder.

--- TomekLinks ---
Accuracy: 0.7454 | F1: 0.7401 | ROC-AUC: 0.8953
✅ Success! Results for VotingClassifier (LR+RFC+SVC) + TomekLinks saved in the '../comparations' folder.

--- RandomOverSampler ---
Accuracy: 0.7351 | F1: 0.7372 | ROC-AUC: 0.8963
✅ Success! Results for VotingClassifier (LR+RFC+SVC) + RandomOverSampler saved in the '../comparations' folder.

--- SMOTE ---
Accuracy: 0.7287 | F1: 0.7310 | ROC-AUC: 0.8945
✅ Success! Results for VotingClassifier (LR+RFC+SVC) + SMOTE saved in the '../comparations' folder.

--- ADASYN ---
Accuracy: 0.7273 | F1: 0.7295 | ROC-AUC: 0.8942
✅ Success! Results for VotingClassifier (LR+RFC+SVC) + 

Hyperparameter Optuna for VotingClassifier

In [ ]:
import optuna
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import f1_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

import sys
sys.path.append("C:\\Users\\alvarodela.herran\\Machine_learning\\Adv_ML\\Data_preprocesing")
from IQRCapper import IQRCapper

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective_voting(trial):

    
    lr_C = trial.suggest_float('lr_C', 0.01, 10.0, log=True)

    
    rfc_n_estimators  = trial.suggest_int('rfc_n_estimators', 50, 300)
    rfc_max_depth     = trial.suggest_int('rfc_max_depth', 3, 20)
    rfc_max_features  = trial.suggest_float('rfc_max_features', 0.3, 1.0)

    
    svc_C      = trial.suggest_float('svc_C', 0.01, 10.0, log=True)
    svc_kernel = trial.suggest_categorical('svc_kernel', ['rbf', 'linear'])

    lr  = LogisticRegression(C=lr_C, max_iter=1000, random_state=42)

    rfc = RandomForestClassifier(
        n_estimators=rfc_n_estimators,
        max_depth=rfc_max_depth,
        max_features=rfc_max_features,
        random_state=42,
        n_jobs=-1
    )

    svc = SVC(
        C=svc_C,
        kernel=svc_kernel,
        probability=True,
        random_state=42
    )

    voting_clf = VotingClassifier(
        estimators=[('lr', lr), ('rfc', rfc), ('svc', svc)],
        voting='soft'
    )

    pipe = Pipeline([
        ('capping_outliers', IQRCapper(factor=1.5)),
        ('imputacion', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler()),
        ('classifier', voting_clf)
    ])

    pipe.fit(X_train, y_train)

    y_pred = pipe.predict(X_test)

    return f1_score(y_test, y_pred, average='weighted')

study_voting = optuna.create_study(direction='maximize')
study_voting.optimize(objective_voting, n_trials=20)

print(f"\nBest F1: {study_voting.best_value:.4f}")
print(f"Best hyperparameters: {study_voting.best_params}")


Best F1: 0.7488
Best hyperparameters: {'lr_C': 1.9412175890645718, 'rfc_n_estimators': 228, 'rfc_max_depth': 14, 'rfc_max_features': 0.4832942517247062, 'svc_C': 8.796832403825867, 'svc_kernel': 'rbf'}


In [5]:
import optuna.visualization as ov

ov.plot_param_importances(study_voting).show()
ov.plot_optimization_history(study_voting).show()
ov.plot_slice(study_voting).show()

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

best = study_voting.best_params

lr  = LogisticRegression(C=best['lr_C'], max_iter=1000, random_state=42)
rfc = RandomForestClassifier(
    n_estimators=best['rfc_n_estimators'],
    max_depth=best['rfc_max_depth'],
    max_features=best['rfc_max_features'],
    random_state=42,
    n_jobs=-1
)
svc = SVC(C=best['svc_C'], kernel=best['svc_kernel'], probability=True, random_state=42)

voting_final = VotingClassifier(
    estimators=[('lr', lr), ('rfc', rfc), ('svc', svc)],
    voting='soft'
)

pipe_voting_final = Pipeline([
    ('capping_outliers', IQRCapper(factor=1.5)),
    ('imputacion', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('classifier', voting_final)
])

pipe_voting_final.fit(X_train, y_train)
y_pred  = pipe_voting_final.predict(X_test)
y_proba = pipe_voting_final.predict_proba(X_test)

accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall    = recall_score(y_test, y_pred, average='weighted')
f1        = f1_score(y_test, y_pred, average='weighted')
roc_auc   = roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted')

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1:        {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")

save_experiment(
    model_name="VotingClassifier Optuna v2 (Final)",
    imbalance_method="Baseline",
    accuracy=accuracy,
    precision=precision,
    recall=recall,
    f1=f1,
    roc_auc=roc_auc
)

Accuracy:  0.7524
Precision: 0.7500
Recall:    0.7524
F1:        0.7488
ROC-AUC:   0.8989
✅ Success! Results for VotingClassifier Optuna v2 (Final) + Baseline saved in the '../comparations' folder.
